# 🐶 Sistema Experto con Incertidumbre para Identificar Razas de Perros

En este cuaderno construyo un **sistema experto** capaz de **sugerir la raza más probable** de un perro a partir de rasgos observables (tamaño, tipo de pelo, orejas, hocico, cola, color, etc.), considerando que las descripciones reales suelen ser **imprecisas** o **parciales**.

---

## 🎯 Objetivo
Dado un conjunto de **hechos con grado de certeza** (por ejemplo: *“orejas paradas” = 0.8*), el sistema infiere una o varias **razas candidatas** y las ordena por **Factor de Certeza (CF)**.

---

## 🧠 Componentes del sistema
- **Base de conocimiento:** reglas tipo **SI (rasgos) ENTONCES (raza)** con un **CF** que indica qué tan representativa es la regla.
- **Memoria de trabajo:** hechos del caso actual (rasgos ingresados) con su incertidumbre.
- **Motor de inferencia:** encadenamiento hacia adelante que:
  - calcula la certeza de los antecedentes (AND → `min`)
  - propaga certeza a conclusiones (CF_antecedentes × CF_regla)
  - combina evidencias múltiples (función `combinar_cf`)
- **Módulo de explicación:** registra qué reglas se activaron y cuánto aportó cada una al resultado.

---

## 🔎 Qué produce el sistema
- Un **ranking** de razas con su CF final (más alto = más probable).
- Una **traza** explicable: “por qué” se recomendó esa raza.

---

## 🧪 Flujo del cuaderno
1. Definir rasgos (hechos) y cómo se ingresan con incertidumbre.
2. Definir reglas y sus factores de certeza.
3. Ejecutar el motor de inferencia.
4. Mostrar ranking + explicación.

> Nota: Los **CF no son probabilidades**, son una medida de **confianza** basada en evidencia y reglas.

In [18]:
# =========================
# IMPORTS Y CONFIGURACIÓN
# =========================
from dataclasses import dataclass
from typing import List, Dict, Tuple

# Configuración del sistema
class Config:
    """Configuración centralizada del sistema experto"""
    # Motor de inferencia
    EPSILON_CONVERGENCIA = 1e-3  # Umbral de cambio para convergencia
    MIN_COINCIDENCIAS_RESPALDO = 2  # Mínimo de coincidencias para modo respaldo
    
    # Ranking y visualización
    TOP_RAZAS_MOSTRAR = 5  # Número de razas a mostrar en resultados
    LONGITUD_BARRA_PROGRESO = 20  # Caracteres en barras de progreso
    
    # Reconocimiento de voz
    TIMEOUT_VOZ = 10  # Segundos de espera para iniciar voz
    PHRASE_TIME_LIMIT = 15  # Duración máxima de grabación
    PAUSE_THRESHOLD = 1.2  # Pausa para detectar fin de frase
    ENERGY_THRESHOLD = 4000  # Umbral de energía de audio
    IDIOMA_RECONOCIMIENTO = "es-ES"  # Código de idioma para Google Speech API

## 📐 Estructuras de Datos

Definición de las clases y tipos de datos fundamentales del sistema experto.

In [19]:
@dataclass
class Regla:
    """
    Representa una regla del sistema experto.
    
    Attributes:
        si: Lista de hechos (rasgos) requeridos para activar la regla
        entonces: Raza resultante si se cumple la regla
        fc: Factor de certeza de la regla (rango: -1 a 1)
            - Positivo: evidencia a favor de la raza
            - Negativo: evidencia en contra de la raza
        id: Identificador único de la regla para trazabilidad
    """
    si: List[str]
    entonces: str
    fc: float
    id: str = ""


## ⚙️ Funciones Auxiliares

Funciones de utilidad para combinar certezas y calcular aportes de reglas.

In [20]:
def combinar_cf(cf1: float, cf2: float) -> float:
    """
    Combina dos factores de certeza según la teoría de Certainty Factors.
    
    Args:
        cf1: Primer factor de certeza (rango: -1 a 1)
        cf2: Segundo factor de certeza (rango: -1 a 1)
    
    Returns:
        Factor de certeza combinado
        
    Casos:
        - Ambos positivos (a favor): refuerzo acumulativo
        - Ambos negativos (en contra): refuerzo negativo
        - Conflicto (uno positivo, uno negativo): promedio ponderado
        
    Fórmula:
        - CF(A,B) = CF(A) + CF(B) × (1 - CF(A))  si ambos ≥ 0
        - CF(A,B) = CF(A) + CF(B) × (1 + CF(A))  si ambos ≤ 0
        - CF(A,B) = (CF(A) + CF(B)) / (1 - min(|CF(A)|, |CF(B)|))  en conflicto
    """
    # Ambos a favor
    if cf1 >= 0 and cf2 >= 0:
        return cf1 + cf2 * (1 - cf1)
    # Ambos en contra
    if cf1 <= 0 and cf2 <= 0:
        return cf1 + cf2 * (1 + cf1)
    # Conflicto
    return (cf1 + cf2) / (1 - min(abs(cf1), abs(cf2)))


def _calcular_aporte_regla(regla: Regla, hechos: Dict[str, float]) -> Tuple[float, float]:
    """
    Calcula el aporte de una regla dados los hechos observados.
    
    Helper function para evitar duplicación de código entre inferir() e inferir_1pasada().
    
    Args:
        regla: Regla a aplicar
        hechos: Diccionario {hecho: CF} con evidencias
    
    Returns:
        Tupla (fc_antecedentes, fc_aporte) donde:
        - fc_antecedentes: CF mínimo de los antecedentes (certeza más débil)
        - fc_aporte: Contribución de esta regla (fc_ant * fc_regla)
    
    Ejemplo:
        >>> regla = Regla(id="R1", si=["grande", "peludo"], entonces="Husky", fc=0.9)
        >>> hechos = {"grande": 0.8, "peludo": 0.9}
        >>> _calcular_aporte_regla(regla, hechos)
        (0.8, 0.72)
    """
    fc_ant = min(hechos[hecho] for hecho in regla.si)
    fc_aporte = fc_ant * regla.fc
    return fc_ant, fc_aporte


## 🔄 Motor de Inferencia

Implementación del encadenamiento hacia adelante con manejo de incertidumbre y modo respaldo para coincidencias parciales.

In [21]:
def inferir(hechos: Dict[str, float], reglas: List[Regla], epsilon: float = 1e-3):
    """
    Motor de inferencia con encadenamiento hacia adelante.
    
    Ejecuta iterativamente todas las reglas cuyas condiciones se cumplan completamente,
    propagando certezas y combinando evidencias hasta convergencia.
    
    Args:
        hechos: Diccionario {nombre_hecho: certeza} con los rasgos observados
        reglas: Lista de reglas del sistema experto
        epsilon: Umbral de cambio para convergencia (default: 1e-3)
    
    Returns:
        Tupla (ranking, logs) donde:
        - ranking: Lista ordenada de tuplas (raza, CF_final) por certeza descendente
        - logs: Diccionario {raza: [detalles_de_activacion]} con trazabilidad
    
    Comportamiento:
        - Solo activa reglas con TODAS sus condiciones presentes en hechos
        - Combina evidencias de múltiples reglas usando combinar_cf()
        - Itera hasta que no haya cambios significativos (< epsilon)
    """
    conclusiones: Dict[str, float] = {}  # raza--> CF final
    logs: Dict[str, List[dict]] = {}   # raza -> lista de aportes

    hubo_cambio = True
    while hubo_cambio:
        hubo_cambio = False

        for regla in reglas:
            if all(hecho in hechos for hecho in regla.si):
                # Usar función helper para calcular aporte
                fc_ant, fc_aporte = _calcular_aporte_regla(regla, hechos)

                prev = conclusiones.get(regla.entonces, 0.0)
                nuevo = combinar_cf(prev, fc_aporte)

                if abs(nuevo - prev) > epsilon:
                    conclusiones[regla.entonces] = nuevo
                    hubo_cambio = True

                logs.setdefault(regla.entonces, []).append({
                    "regla": regla.id,
                    "si": regla.si,
                    "fc_antecedentes": round(fc_ant, 4),
                    "fc_regla": regla.fc,
                    "fc_aporte": round(fc_aporte, 4),
                    "cf_prev": round(prev, 4),
                    "cf_nuevo": round(nuevo, 4),
                })

    ranking = sorted(conclusiones.items(), key=lambda x: x[1], reverse=True)
    return ranking, logs


In [22]:
# ============================================================================
# BASE DE CONOCIMIENTO AMPLIADA - RAZAS DE PERROS
# ============================================================================
# Nota: Los CF reflejan:
#   - CF alto (0.85+): característica muy típica de la raza
#   - CF medio (0.70-0.84): característica frecuente
#   - CF bajo (0.50-0.69): característica moderada
#   - CF negativo: descarta la raza (anti-evidencia)
# ============================================================================

REGLAS = [
    # ========== PASTOR ALEMÁN ==========
    Regla(["tamano_grande", "pelo_medio", "orejas_paradas", "mascara_cara", "cola_poblada"], "Pastor Aleman", 0.92, "R1"),
    Regla(["tamano_grande", "color_bicolor", "cuerpo_rectangulado"], "Pastor Aleman", 0.88, "R2"),
    Regla(["tamano_grande", "hocico_largo", "expresion_inteligente"], "Pastor Aleman", 0.80, "R3"),
    Regla(["pelo_rizado"], "Pastor Aleman", -0.85, "R_NEG_1"),

    # ========== HUSKY SIBERIANO ==========
    Regla(["tamano_grande", "pelo_largo", "ojos_azules", "orejas_paradas", "cola_poblada"], "Husky Siberiano", 0.95, "R4"),
    Regla(["tamano_grande", "color_blanco_gris"], "Husky Siberiano", 0.87, "R5"),
    Regla(["ojos_azules", "orejas_paradas"], "Husky Siberiano", 0.82, "R6"),
    Regla(["pelo_corto"], "Husky Siberiano", -0.70, "R_NEG_2"),

    # ========== BEAGLE ==========
    Regla(["tamano_mediano", "pelo_corto", "orejas_caidas", "tricolor"], "Beagle", 0.90, "R7"),
    Regla(["tamano_mediano", "hocico_puntiagudo", "nariz_negra"], "Beagle", 0.80, "R8"),
    Regla(["tamano_pequeno", "tricolor", "orejas_caidas"], "Beagle", 0.75, "R9"),
    Regla(["pelo_rizado"], "Beagle", -0.65, "R_NEG_3"),

    # ========== GOLDEN RETRIEVER ==========
    Regla(["tamano_grande", "dorado", "pelo_medio", "orejas_caidas"], "Golden Retriever", 0.93, "R10"),
    Regla(["tamano_grande", "color_dorado", "ojos_oscuros"], "Golden Retriever", 0.89, "R11"),
    Regla(["cola_enrollada"], "Golden Retriever", -0.75, "R_NEG_4"),
    Regla(["pelaje_abundante", "orejas_caidas", "expresion_amigable"], "Golden Retriever", 0.78, "R12"),

    # ========== LABRADOR RETRIEVER ==========
    Regla(["tamano_grande", "pelo_corto", "dorado", "orejas_caidas"], "Labrador Retriever", 0.91, "R13"),
    Regla(["tamano_grande", "color_negro", "orejas_caidas"], "Labrador Retriever", 0.88, "R14"),
    Regla(["tamano_grande", "musculoso", "cola_gruesa"], "Labrador Retriever", 0.82, "R15"),
    Regla(["pelo_rizado"], "Labrador Retriever", -0.80, "R_NEG_5"),
    Regla(["hocico_corto"], "Labrador Retriever", -0.60, "R_NEG_6"),

    # ========== ROTTWEILER ==========
    Regla(["tamano_grande", "pelo_corto", "robusto", "negro_y_fuego"], "Rottweiler", 0.94, "R16"),
    Regla(["tamano_grande", "musculoso", "mandibula_fuerte"], "Rottweiler", 0.85, "R17"),
    Regla(["color_negro", "marcas_fuego", "orejas_triangulares"], "Rottweiler", 0.88, "R18"),

    # ========== DACHSHUND ==========
    Regla(["tamano_pequeno", "pelo_largo", "cuerpo_alargado", "orejas_caidas"], "Dachshund", 0.91, "R19"),
    Regla(["cuerpo_alargado", "patas_cortas", "tamano_pequeno"], "Dachshund", 0.89, "R20"),
    Regla(["orejas_muy_largas", "patas_muy_cortas"], "Dachshund", 0.87, "R21"),
    Regla(["tamano_grande"], "Dachshund", -0.90, "R_NEG_7"),

    # ========== BULLDOG FRANCÉS ==========
    Regla(["tamano_pequeno", "hocico_corto", "orejas_paradas", "cuerpo_robusto"], "Bulldog Frances", 0.93, "R22"),
    Regla(["tamano_pequeno", "cara_plana", "arrugas_cara"], "Bulldog Frances", 0.90, "R23"),
    Regla(["orejas_murciélago", "ojos_salientes"], "Bulldog Frances", 0.85, "R24"),
    Regla(["pelo_largo"], "Bulldog Frances", -0.80, "R_NEG_8"),
    Regla(["tamano_grande"], "Bulldog Frances", -0.95, "R_NEG_9"),

    # ========== POODLE ==========
    Regla(["tamano_mediano", "pelo_rizado", "orejas_caidas", "color_uniforme"], "Poodle", 0.92, "R25"),
    Regla(["pelo_rizado", "hipoalergenico"], "Poodle", 0.88, "R26"),
    Regla(["pelaje_abundante", "pelo_rizado"], "Poodle", 0.90, "R27"),
    Regla(["pelo_liso"], "Poodle", -0.75, "R_NEG_10"),

    # ========== CHIHUAHUA ==========
    Regla(["tamano_muy_pequeno", "cabeza_grande", "ojos_grandes"], "Chihuahua", 0.89, "R28"),
    Regla(["tamano_muy_pequeno", "orejas_puntiagudas", "cola_enrollada"], "Chihuahua", 0.85, "R29"),
    Regla(["peso_bajo", "ojos_saltones"], "Chihuahua", 0.82, "R30"),
    Regla(["tamano_grande"], "Chihuahua", -0.95, "R_NEG_11"),

    # ========== BULLDOG INGLÉS ==========
    Regla(["tamano_mediano", "cara_plana", "mandibula_inferior_prominente"], "Bulldog Ingles", 0.91, "R31"),
    Regla(["tamano_mediano", "arrugas_abundantes", "cuerpo_bajo"], "Bulldog Ingles", 0.88, "R32"),
    Regla(["orejas_caidas", "cabeza_grande"], "Bulldog Ingles", 0.80, "R33"),

    # ========== PASTOR INGLÉS ANTIGUO ==========
    Regla(["tamano_grande", "pelaje_largo", "bicolor", "pelo_ondulado"], "Pastor Ingles Antiguo", 0.90, "R34"),
    Regla(["tamano_grande", "ojos_cubiertos", "pelaje_abundante"], "Pastor Ingles Antiguo", 0.85, "R35"),

    # ========== BOXER ==========
    Regla(["tamano_grande", "musculoso", "hocico_corto", "mandibula_pronunciada"], "Boxer", 0.88, "R36"),
    Regla(["tamano_grande", "color_atigrado", "orejas_caidas_naturales"], "Boxer", 0.82, "R37"),
    Regla(["energico", "cuerpo_atlético"], "Boxer", 0.75, "R38"),

    # ========== SCHNAUZER ==========
    Regla(["tamano_mediano", "bigote_caracteristico", "barba_larga"], "Schnauzer", 0.92, "R39"),
    Regla(["color_gris_sal_pimienta", "orejas_triangulares"], "Schnauzer", 0.86, "R40"),
    Regla(["pelo_duro", "bigotes_largos"], "Schnauzer", 0.88, "R41"),

    # ========== SHAR PEI ==========
    Regla(["tamano_mediano", "arrugas_excesivas", "piel_suelta"], "Shar Pei", 0.94, "R42"),
    Regla(["color_azul", "orejas_pequeñas", "cola_enrollada"], "Shar Pei", 0.89, "R43"),
    Regla(["orejas_muy_pequeñas", "lengua_azul"], "Shar Pei", 0.90, "R44"),

    # ========== COCKER SPANIEL ==========
    Regla(["tamano_mediano", "pelo_largo_ondulado", "orejas_caidas"], "Cocker Spaniel", 0.89, "R45"),
    Regla(["color_dorado_oscuro", "orejas_caidas_largas"], "Cocker Spaniel", 0.85, "R46"),
    Regla(["pelaje_sedoso", "ojos_dulces"], "Cocker Spaniel", 0.80, "R47"),

    # ========== DALMATIAN ==========
    Regla(["tamano_grande", "color_blanco_manchas_negras", "orejas_caidas"], "Dalmatian", 0.95, "R48"),
    Regla(["manchas_oscuras", "pelaje_corto"], "Dalmatian", 0.90, "R49"),
    Regla(["color_uniforme"], "Dalmatian", -0.85, "R_NEG_12"),

    # ========== PASTOR GANADERO AUSTRALIANO ==========
    Regla(["tamano_mediano", "color_bicolor_jaspeado", "orejas_paradas"], "Pastor Ganadero Australiano", 0.88, "R50"),
    Regla(["resistente", "inteligente_activo"], "Pastor Ganadero Australiano", 0.78, "R51"),

    # ========== CORGI ==========
    Regla(["tamano_pequeno", "patas_cortas", "cuerpo_largo"], "Corgi", 0.87, "R52"),
    Regla(["orejas_paradas", "cara_alerta"], "Corgi", 0.80, "R53"),
    Regla(["patas_cortas", "pesar_bajo"], "Corgi", 0.82, "R54"),

    # ========== PASTOR BELGA ==========
    Regla(["tamano_grande", "pelo_largo", "color_negro", "orejas_paradas"], "Pastor Belga", 0.89, "R55"),
    Regla(["color_negro_fuego", "musculoso"], "Pastor Belga", 0.85, "R56"),

    # ========== COLLIE ==========
    Regla(["tamano_grande", "hocico_largo_puntiagudo", "orejas_paradas"], "Collie", 0.86, "R57"),
    Regla(["pelaje_largo", "color_rojo_blanco"], "Collie", 0.82, "R58"),

    # ========== SHIBA INU ==========
    Regla(["tamano_pequeno", "orejas_paradas", "cola_enrollada"], "Shiba Inu", 0.88, "R59"),
    Regla(["color_rojo", "cara_coqueta"], "Shiba Inu", 0.84, "R60"),

    # ========== AKITA ==========
    Regla(["tamano_grande", "orejas_paradas", "cola_enrollada", "musculoso"], "Akita", 0.89, "R61"),
    Regla(["color_rojo_blanco", "expresion_seria"], "Akita", 0.83, "R62"),
]


In [23]:
# ============================================================================
# DICCIONARIO DE RASGOS DISPONIBLES CON CERTEZA
# ============================================================================
# Nota: Cada rasgo puede ingresarse con certeza entre 0 y 1
#       - 1.0 = muy seguro del rasgo
#       - 0.7-0.9 = bastante seguro
#       - 0.5 = dudoso/ambiguo
#       - 0.0-0.4 = muy incierto

RASGOS_DISPONIBLES = {
    # TAMAÑO
    "tamano_muy_pequeno": "Perro pequeño (< 5 kg), ej: Chihuahua",
    "tamano_pequeno": "Perro pequeño (5-10 kg), ej: Bulldog Francés",
    "tamano_mediano": "Perro mediano (10-30 kg), ej: Beagle",
    "tamano_grande": "Perro grande (>30 kg), ej: Pastor Alemán",
    
    # CABEZA Y HOCICO
    "cabeza_grande": "Cabeza proporcionalmente grande",
    "hocico_largo": "Hocico/morro largo y puntiagudo",
    "hocico_corto": "Cara plana, hocico muy corto",
    "cara_plana": "Cráneo achatado",
    "hocico_puntiagudo": "Punta de hocico fina",
    "mandibula_fuerte": "Mandíbula/quijada muy pronunciada",
    "mandibula_inferior_prominente": "Quijada inferior saliente",
    "mascara_cara": "Máscara oscura en la cara",
    "arrugas_cara": "Pliegues/arrugas en la cara",
    "arrugas_abundantes": "Muchas arrugas, típico de ciertos tipos",
    "ojos_grandes": "Ojos notoriamente grandes",
    "ojos_saltones": "Ojos prominentes",
    "ojos_azules": "Ojos de color azul",
    "ojos_oscuros": "Ojos marrones o negros",
    
    # OREJAS
    "orejas_paradas": "Orejas erguidas hacia arriba",
    "orejas_caidas": "Orejas colgantes",
    "orejas_triangulares": "Forma triangular de las orejas",
    "orejas_muy_largas": "Orejas excepcionalmente largas",
    "orejas_pequenas": "Orejas pequeñas",
    "orejas_muy_pequeñas": "Orejas diminutas",
    "orejas_murciélago": "Orejas paradas en forma de alas",
    "orejas_caidas_largas": "Orejas largas y colgantes",
    
    # PELAJE Y COLOR
    "pelo_largo": "Pelaje largo",
    "pelo_medio": "Pelaje de longitud media",
    "pelo_corto": "Pelaje muy corto",
    "pelo_rizado": "Textura rizada u ondulada",
    "pelo_ondulado": "Pelaje con ondas",
    "pelo_duro": "Pelaje duro, áspero al tacto",
    "pelaje_sedoso": "Pelaje suave y sedoso",
    "pelaje_abundante": "Pelaje muy denso y voluminoso",
    "hipoalergenico": "Pelaje que no suelta pelos fácilmente",
    "color_negro": "Color negro dominante",
    "color_blanco": "Color blanco",
    "color_dorado": "Color dorado/oro",
    "dorado": "Tonalidad dorada",
    "color_rojo": "Color rojo o marrón rojizo",
    "color_blanco_gris": "Combinación blanco-gris",
    "color_blanco_manchas_negras": "Blanco con manchas negras",
    "manchas_oscuras": "Manchas negras o marrones",
    "color_negro_fuego": "Negro con marcas fuego/doradas",
    "negro_y_fuego": "Negro con puntos color fuego",
    "color_uniforme": "Color sólido sin variaciones",
    "color_atigrado": "Patrón de rayas/manchas",
    "color_bicolor": "Dos colores principales",
    "bicolor": "Dos colores distintos",
    "color_bicolor_jaspeado": "Dos colores mezclados",
    "tricolor": "Tres colores (típico: blanco, negro, fuego)",
    "color_dorado_oscuro": "Dorado profundo",
    "color_azul": "Tonalidad azulada del pelaje",
    "color_gris_sal_pimienta": "Gris moteado",
    "color_rojo_blanco": "Rojo/marrón con blanco",
    
    # CUERPO Y ESTRUCTURA
    "cuerpo_alargado": "Cuerpo largo respecto a altura",
    "cuerpo_bajo": "Patas cortas, cuerpo bajo",
    "cuerpo_rectangulado": "Forma rectangular, más largo que alto",
    "cuerpo_robusto": "Cuerpo compacto y musculoso",
    "musculoso": "Musculatura muy pronunciada",
    "patas_cortas": "Patas notoriamente cortas",
    "patas_muy_cortas": "Patas excepcionalmente cortas",
    "pesar_bajo": "Peso corporal bajo",
    "peso_bajo": "Muy ligero",
    "cuerpo_atlético": "Complexión deportiva",
    
    # COLA
    "cola_poblada": "Cola con pelaje abundante",
    "cola_gruesa": "Cola gruesa y fuerte",
    "cola_enrollada": "Cola rizada o enrollada",
    "cola_recta": "Cola recta",
    
    # PERSONALIDAD/COMPORTAMIENTO (observables)
    "energico": "Comportamiento muy activo/energético",
    "inteligente_activo": "Inteligencia activa, comportamiento alerta",
    "expresion_inteligente": "Mirada inteligente",
    "expresion_amigable": "Expresión dulce/amigable",
    "expresion_seria": "Expresión seria/seria",
    "cara_coqueta": "Expresión traviesa/coqueta",
    "ojos_dulces": "Ojos expresivos dulces",
    "resistente": "Comportamiento resistente/ágil",
    "lengua_azul": "Lengua azulada",
    "nariz_negra": "Nariz negra",
}

# EJEMPLO DE HECHOS (entrada del usuario con incertidumbre)
hechos_ejemplo_1 = {
    "tamano_grande": 0.9,
    "pelo_largo": 0.8,
    "ojos_azules": 0.7,
    "orejas_paradas": 0.9,
    "cola_poblada": 0.8
}

# Otro ejemplo: Bulldog Francés
hechos_ejemplo_2 = {
    "tamano_pequeno": 0.95,
    "cara_plana": 0.9,
    "hocico_corto": 0.88,
    "orejas_paradas": 0.85,
    "arrugas_cara": 0.80,
    "cuerpo_robusto": 0.85,
    "ojos_saltones": 0.75,
}

# Otro ejemplo: Dalmatian
hechos_ejemplo_3 = {
    "tamano_grande": 0.90,
    "color_blanco_manchas_negras": 0.95,
    "manchas_oscuras": 0.92,
    "pelaje_corto": 0.88,
    "orejas_caidas": 0.70,
    "musculoso": 0.80,
}

# Usar uno de estos ejemplos:
hechos = hechos_ejemplo_1


## 🎤 Módulo de Reconocimiento de Voz

Extracción de rasgos desde descripciones verbales usando reconocimiento de voz y procesamiento de lenguaje natural.

In [24]:
# =========================
# ENTRADA POR VOZ (AUDIO -> HECHOS CON CF -> INFERENCIA)
# =========================
import re
import unicodedata

try:
    import speech_recognition as sr
    SR_DISPONIBLE = True
except Exception:
    sr = None
    SR_DISPONIBLE = False


def normalizar_texto(texto: str) -> str:
    """
    Normaliza texto para facilitar coincidencias de patrones regex.
    
    Transformaciones aplicadas:
    1. Convierte a minúsculas
    2. Elimina espacios al inicio/final
    3. Normaliza caracteres Unicode (NFD)
    4. Elimina marcas diacríticas/acentos
    
    Args:
        texto: Cadena de texto a normalizar (puede ser None)
    
    Returns:
        Texto normalizado sin acentos, en minúsculas, sin espacios extra
    
    Ejemplo:
        >>> normalizar_texto("Perro GRANDE con orejas paradas")
        'perro grande con orejas paradas'
    """
    texto = (texto or "").lower().strip()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(ch for ch in texto if unicodedata.category(ch) != "Mn")
    return texto


# Cada patrón mapea texto hablado -> (hecho, CF sugerido)
PATRONES_VOZ = {
    "tamano_grande": [
        (r"\b(grande|enorme|gigante)\b", 0.90),
    ],
    "tamano_mediano": [
        (r"\b(mediano|tamano medio|tamano mediano)\b", 0.85),
    ],
    "tamano_pequeno": [
        (r"\b(pequeno|chico|mini)\b", 0.90),
    ],
    "pelo_largo": [
        (r"\b(pelo largo|pelaje largo)\b", 0.90),
    ],
    "pelo_medio": [
        (r"\b(pelo medio|pelaje medio)\b", 0.80),
    ],
    "pelo_corto": [
        (r"\b(pelo corto|pelaje corto)\b", 0.90),
    ],
    "pelo_rizado": [
        (r"\b(pelo rizado|rizado|enchinado)\b", 0.92),
    ],
    "orejas_paradas": [
        (r"\b(orejas paradas|orejas erguidas)\b", 0.90),
    ],
    "orejas_caidas": [
        (r"\b(orejas caidas|orejas largas caidas)\b", 0.90),
    ],
    "ojos_azules": [
        (r"\b(ojos azules|ojo azul|azules)\b", 0.88),
    ],
    "cola_poblada": [
        (r"\b(cola poblada|cola peluda)\b", 0.85),
    ],
    "cola_enrollada": [
        (r"\b(cola enrollada|cola rizada)\b", 0.88),
    ],
    "dorado": [
        (r"\b(dorado|golden)\b", 0.90),
    ],
    "color_negro": [
        (r"\b(negro|negra)\b", 0.82),
    ],
    "tricolor": [
        (r"\b(tricolor|tres colores)\b", 0.90),
    ],
    "negro_y_fuego": [
        (r"\b(negro y fuego|marcas fuego)\b", 0.92),
    ],
    "cuerpo_alargado": [
        (r"\b(cuerpo alargado|largo de cuerpo)\b", 0.88),
    ],
    "hocico_corto": [
        (r"\b(hocico corto|cara chata|cara plana)\b", 0.90),
    ],
    "mascara_cara": [
        (r"\b(mascara en la cara|cara oscura)\b", 0.85),
    ],
    "robusto": [
        (r"\b(robusto|fuerte|ancho)\b", 0.82),
    ],
    "musculoso": [
        (r"\b(musculoso|musculatura marcada)\b", 0.84),
    ],
}


def extraer_hechos_desde_texto(texto: str) -> Dict[str, float]:
    texto_n = normalizar_texto(texto)
    hechos_audio: Dict[str, float] = {}

    for hecho, patrones in PATRONES_VOZ.items():
        for patron, cf_sugerido in patrones:
            if re.search(patron, texto_n):
                hechos_audio[hecho] = max(hechos_audio.get(hecho, 0.0), cf_sugerido)

    return hechos_audio


def reconocer_voz_es(timeout=15, phrase_time_limit=20):
    """
    Reconocimiento de voz simple y confiable usando listen() en lugar de listen_in_background().
    Más compatible con Jupyter y evita problemas de threading.
    
    Args:
        timeout: Tiempo máximo de espera para que empiece a hablar (segundos)
        phrase_time_limit: Tiempo máximo de grabación total (segundos)
    """
    if not SR_DISPONIBLE:
        print("❌ No está instalado speech_recognition.")
        print("   Instala: pip install SpeechRecognition")
        print("   Si hace falta micrófono en Windows: pip install pipwin && pipwin install pyaudio")
        return None

    recognizer = sr.Recognizer()
    # Configuración optimizada para español
    recognizer.energy_threshold = 4000  # Ajustar según ruido ambiente
    recognizer.dynamic_energy_threshold = True
    recognizer.pause_threshold = 1.2  # Pausa para considerar fin de frase
    recognizer.phrase_threshold = 0.3
    recognizer.non_speaking_duration = 0.8

    try:
        with sr.Microphone() as source:
            print("🎤 Describe al perro (tamaño, pelo, orejas, color, etc.)...")
            print("   📊 Calibrando micrófono...")
            
            # Ajustar por ruido ambiente
            recognizer.adjust_for_ambient_noise(source, duration=1.5)
            
            print(f"   🔴 GRABANDO (máximo {phrase_time_limit}s)... ¡Habla ahora!")
            print("   💬 Ejemplo: 'es un perro grande, dorado, con pelo largo'")
            
            # Escuchar UNA SOLA VEZ con límite de tiempo (más simple y confiable)
            audio = recognizer.listen(
                source,
                timeout=timeout,
                phrase_time_limit=phrase_time_limit
            )
            
            print("   ⏸️  Procesando audio...")

        # Reconocer usando Google Speech API
        try:
            texto = recognizer.recognize_google(audio, language="es-ES")
            
            if texto and texto.strip():
                print(f"   ✅ Audio capturado correctamente\n")
                print(f"📝 Texto reconocido: \"{texto}\"")
                return texto.strip()
            else:
                print("🤷 El audio estaba vacío o no se entendió.")
                return None
                
        except sr.UnknownValueError:
            print("🤷 No se pudo entender el audio. Intenta hablar más claro o más cerca del micrófono.")
            return None
        except sr.RequestError as e:
            print(f"🌐 Error de conexión con Google Speech API: {e}")
            print("   Verifica tu conexión a Internet.")
            return None

    except sr.WaitTimeoutError:
        print(f"⏱️ No se detectó ninguna voz en {timeout} segundos.")
        print("   Asegúrate de que el micrófono esté funcionando y habla más fuerte.")
        return None
    except KeyboardInterrupt:
        print("\n⏹️ Grabación cancelada por el usuario.")
        return None
    except OSError as e:
        print(f"❌ Error de hardware/micrófono: {e}")
        print("\n💡 Soluciones:")
        print("   1. Verifica que el micrófono esté conectado")
        print("   2. Comprueba permisos de micrófono en Windows")
        print("   3. Prueba cerrar otras aplicaciones que usen el micrófono")
        return None
    except Exception as e:
        print(f"⚠️ Error inesperado: {e}")
        import traceback
        traceback.print_exc()
        return None


def diagnosticar_desde_audio(usando_1pasada: bool = False, top_n: int = 5):
    """
    Pipeline completo de diagnóstico por voz: captura audio → extrae hechos → infiere razas.
    
    Flujo:
    1. Captura audio del micrófono con reconocer_voz_es()
    2. Convierte texto a hechos con extraer_hechos_desde_texto()
    3. Aplica motor de inferencia (inferir o inferir_1pasada)
    4. Muestra top N razas más probables
    
    Args:
        usando_1pasada: Si True, usa inferir_1pasada() con modo respaldo; si False usa inferir() clásico
        top_n: Cantidad de razas a mostrar en el ranking
    
    Returns:
        None (imprime resultados directamente)
    
    Ejemplo de uso:
        >>> diagnosticar_desde_audio(usando_1pasada=True, top_n=5)
        🎤 Habla ahora (presiona Enter cuando termines)...
        📝 Texto reconocido: "perro grande con pelo largo"
        🔎 Hechos detectados...
        📊 Top razas inferidas:
        1. Husky Siberiano -> CF=0.720
    """
    texto = reconocer_voz_es()
    if not texto:
        return None

    hechos_audio = extraer_hechos_desde_texto(texto)
    if not hechos_audio:
        print("❌ No se detectaron rasgos válidos en el audio.")
        return None

    print("\n🔎 Hechos detectados (con certeza):")
    for h, cf in sorted(hechos_audio.items(), key=lambda x: x[1], reverse=True):
        print(f"- {h}: {cf:.2f}")

    if usando_1pasada and "inferir_1pasada" in globals():
        ranking, logs = inferir_1pasada(hechos_audio, REGLAS, usar_respaldo=True, min_coincidencias=2)
    else:
        ranking, logs = inferir(hechos_audio, REGLAS)
        ranking = [(raza, cf) for raza, cf in ranking if cf > 0]

    print("\n📊 Top razas inferidas:")
    if not ranking:
        print("- Diagnóstico no concluyente")
        print("  Sugerencia: describe al menos tamaño + pelo + orejas o color.")
        return None

    for i, (raza, cf) in enumerate(ranking[:top_n], 1):
        print(f"{i}. {raza} -> CF={cf:.3f}")

    mejor_raza, mejor_cf = ranking[0]
    print(f"\n✅ Recomendación principal: {mejor_raza} (CF={mejor_cf:.3f})")

    return {
        "texto": texto,
        "hechos": hechos_audio,
        "ranking": ranking,
        "logs": logs,
    }


In [25]:
ranking, logs = inferir(hechos, REGLAS)

print("Top razas:")
for raza, cf in ranking[:5]:
    print(f"- {raza}: {cf:.3f}")

print("\nExplicación de la mejor:")
mejor = ranking[0][0] if ranking else None
if mejor:
    for t in logs[mejor]:
        print(t)
else:
    print("No se encontraron coincidencias.")

Top razas:
- Husky Siberiano: 0.999
- Bulldog Frances: -0.999
- Chihuahua: -1.000
- Dachshund: -1.000

Explicación de la mejor:
{'regla': 'R4', 'si': ['tamano_grande', 'pelo_largo', 'ojos_azules', 'orejas_paradas', 'cola_poblada'], 'fc_antecedentes': 0.7, 'fc_regla': 0.95, 'fc_aporte': 0.665, 'cf_prev': 0.0, 'cf_nuevo': 0.665}
{'regla': 'R6', 'si': ['ojos_azules', 'orejas_paradas'], 'fc_antecedentes': 0.7, 'fc_regla': 0.82, 'fc_aporte': 0.574, 'cf_prev': 0.665, 'cf_nuevo': 0.8573}
{'regla': 'R4', 'si': ['tamano_grande', 'pelo_largo', 'ojos_azules', 'orejas_paradas', 'cola_poblada'], 'fc_antecedentes': 0.7, 'fc_regla': 0.95, 'fc_aporte': 0.665, 'cf_prev': 0.8573, 'cf_nuevo': 0.9522}
{'regla': 'R6', 'si': ['ojos_azules', 'orejas_paradas'], 'fc_antecedentes': 0.7, 'fc_regla': 0.82, 'fc_aporte': 0.574, 'cf_prev': 0.9522, 'cf_nuevo': 0.9796}
{'regla': 'R4', 'si': ['tamano_grande', 'pelo_largo', 'ojos_azules', 'orejas_paradas', 'cola_poblada'], 'fc_antecedentes': 0.7, 'fc_regla': 0.95, 'fc_a

## ✅ Tests (Casos de prueba)

En esta sección se ejecutan **casos de prueba** para validar que el sistema experto:
- identifica correctamente razas en escenarios claros,
- maneja incertidumbre (CF parciales),
- y responde de forma coherente ante conflictos o información insuficiente.

### Todo cambio debe ser testeado 

In [26]:
# =========================
# TESTS (casos de prueba)
# =========================
# Incluye:
# 1) casos funcionales clásicos
# 2) pruebas unitarias de ranking y respaldo por coincidencias parciales

def inferir_1pasada(hechos, reglas, usar_respaldo=False, min_coincidencias=2):
    """
    Motor de inferencia de una sola pasada con modo de respaldo por coincidencia parcial.
    
    Algoritmo:
    1. Aplica todas las reglas POSITIVAS (FC > 0) cuyos antecedentes están completos
    2. Aplica reglas NEGATIVAS (FC < 0) solo a razas ya confirmadas con FC > 0
    3. Si no hay resultados, activa modo RESPALDO: considera reglas con coincidencia parcial
       usando cobertura (ratio de antecedentes presentes) para ajustar el FC
    
    Args:
        hechos: Diccionario {hecho: CF} con evidencias observadas
        reglas: Lista de Regla con conocimiento del dominio
        usar_respaldo: Si True, aplica inferencia parcial cuando no hay reglas completas
        min_coincidencias: Mínimo de antecedentes presentes para considerar regla parcial
    
    Returns:
        Tupla (ranking, logs):
        - ranking: Lista [(raza, CF)] ordenada por CF descendente
        - logs: Dict {raza: [pasos de inferencia]} para trazabilidad
    
    Diferencias con inferir():
        - inferir(): Múltiples iteraciones hasta convergencia (forward chaining completo)
        - inferir_1pasada(): Una sola iteración + modo respaldo opcional (más rápido, menos exhaustivo)
    
    Ejemplo:
        >>> ranking, logs = inferir_1pasada(
        ...     hechos={"tamano_grande": 0.9, "pelo_largo": 0.8},
        ...     reglas=REGLAS,
        ...     usar_respaldo=True,
        ...     min_coincidencias=2
        ... )
        >>> ranking[0]
        ('Husky Siberiano', 0.648)
    """
    conclusiones = {}
    logs = {}

    # PASO 1: Aplicar solo reglas POSITIVAS primero
    for regla in reglas:
        if regla.fc > 0 and all(hecho in hechos for hecho in regla.si):
            # Usar función helper para calcular aporte
            fc_ant, fc_aporte = _calcular_aporte_regla(regla, hechos)

            prev = conclusiones.get(regla.entonces, 0.0)
            nuevo = combinar_cf(prev, fc_aporte)

            conclusiones[regla.entonces] = nuevo
            logs.setdefault(regla.entonces, []).append({
                "regla": regla.id,
                "si": regla.si,
                "si_coincidencias": regla.si,
                "fc_antecedentes": round(fc_ant, 4),
                "fc_regla": regla.fc,
                "fc_aporte": round(fc_aporte, 4),
                "cf_prev": round(prev, 4),
                "cf_nuevo": round(nuevo, 4),
                "metodo": "regla_completa",
            })

    # PASO 2: Aplicar reglas NEGATIVAS solo a razas que ya tienen CF > 0
    for regla in reglas:
        if regla.fc < 0 and all(hecho in hechos for hecho in regla.si):
            if regla.entonces in conclusiones and conclusiones[regla.entonces] > 0:
                # Usar función helper para calcular aporte
                fc_ant, fc_aporte = _calcular_aporte_regla(regla, hechos)

                prev = conclusiones[regla.entonces]
                nuevo = combinar_cf(prev, fc_aporte)

                conclusiones[regla.entonces] = nuevo
                logs[regla.entonces].append({
                    "regla": regla.id,
                    "si": regla.si,
                    "si_coincidencias": regla.si,
                    "fc_antecedentes": round(fc_ant, 4),
                    "fc_regla": regla.fc,
                    "fc_aporte": round(fc_aporte, 4),
                    "cf_prev": round(prev, 4),
                    "cf_nuevo": round(nuevo, 4),
                    "metodo": "regla_negativa",
                })

    # Solo retornar razas con CF > 0
    ranking = sorted([(r, cf) for r, cf in conclusiones.items() if cf > 0], key=lambda x: x[1], reverse=True)

    # RESPALDO: si no hubo reglas completas, estimar por coincidencia parcial
    if not ranking and usar_respaldo:
        puntajes_respaldo = {}
        for regla in reglas:
            if regla.fc <= 0:
                continue

            coincidencias = [hecho for hecho in regla.si if hecho in hechos]
            if len(coincidencias) < min_coincidencias:
                continue

            # Calcular aporte con hechos parciales y ajustar por cobertura
            hechos_parciales = {hecho: hechos[hecho] for hecho in coincidencias}
            fc_ant = min(hechos_parciales.values())
            cobertura = len(coincidencias) / len(regla.si)
            fc_aporte = fc_ant * regla.fc * cobertura

            prev = puntajes_respaldo.get(regla.entonces, 0.0)
            nuevo = combinar_cf(prev, fc_aporte)
            puntajes_respaldo[regla.entonces] = nuevo

            logs.setdefault(regla.entonces, []).append({
                "regla": f"RESP_{regla.id}",
                "si": regla.si,
                "si_coincidencias": coincidencias,
                "fc_antecedentes": round(fc_ant, 4),
                "cobertura": round(cobertura, 4),
                "fc_regla": regla.fc,
                "fc_aporte": round(fc_aporte, 4),
                "cf_prev": round(prev, 4),
                "cf_nuevo": round(nuevo, 4),
                "metodo": "respaldo_parcial",
            })

        ranking = sorted([(r, cf) for r, cf in puntajes_respaldo.items() if cf > 0], key=lambda x: x[1], reverse=True)

    return ranking, logs


TESTS = [
    {
        "nombre": "1) Husky claro",
        "hechos": {"tamano_grande":0.9,"pelo_largo":0.85,"ojos_azules":0.8,"orejas_paradas":0.9,"cola_poblada":0.8},
        "top_esperado": "Husky Siberiano",
    },
    {
        "nombre": "2) Pastor Alemán claro",
        "hechos": {"tamano_grande":0.9,"pelo_medio":0.8,"orejas_paradas":0.9,"mascara_cara":0.85,"cola_poblada":0.8},
        "top_esperado": "Pastor Aleman",
    },
    {
        "nombre": "3) Beagle claro",
        "hechos": {"tamano_mediano":0.9,"pelo_corto":0.9,"orejas_caidas":0.95,"tricolor":0.8},
        "top_esperado": "Beagle",
    },
    {
        "nombre": "4) Golden claro",
        "hechos": {"tamano_grande":0.9,"dorado":0.95,"pelo_medio":0.85,"orejas_caidas":0.8},
        "top_esperado": "Golden Retriever",
    },
    {
        "nombre": "5) Rottweiler claro",
        "hechos": {"tamano_grande":0.9,"pelo_corto":0.9,"robusto":0.85,"negro_y_fuego":0.8},
        "top_esperado": "Rottweiler",
    },
    {
        "nombre": "6) Dachshund claro",
        "hechos": {"tamano_pequeno":0.9,"pelo_largo":0.8,"cuerpo_alargado":0.9,"orejas_caidas":0.85},
        "top_esperado": "Dachshund",
    },
    {
        "nombre": "7) Bulldog Francés claro",
        "hechos": {"tamano_pequeno":0.95,"hocico_corto":0.9,"orejas_paradas":0.8,"cuerpo_robusto":0.85},
        "top_esperado": "Bulldog Frances",
    },
    {
        "nombre": "8) Poodle claro (con color_uniforme)",
        "hechos": {"tamano_mediano":0.85,"pelo_rizado":0.95,"orejas_caidas":0.9,"color_uniforme":0.8},
        "top_esperado": "Poodle",
    },
    {
        "nombre": "9) Labrador claro",
        "hechos": {"tamano_grande":0.9,"pelo_corto":0.85,"dorado":0.8,"orejas_caidas":0.85},
        "top_esperado": "Labrador Retriever",
    },
    {
        "nombre": "10) Conflicto Labrador (rasgos de labrador + pelo_rizado)",
        "hechos": {"tamano_grande":0.9,"pelo_corto":0.85,"dorado":0.8,"orejas_caidas":0.85,"pelo_rizado":0.7},
        "top_esperado": "Labrador Retriever",
    },
    {
        "nombre": "11) Ambiguo: Poodle vs Dachshund (+ Labrador negativo por rizado)",
        "hechos": {"tamano_pequeno":0.9,"pelo_largo":0.7,"cuerpo_alargado":0.8,"orejas_caidas":0.8,"pelo_rizado":0.7},
        "top_esperado": "Dachshund",
    },
    {
        "nombre": "12) Insuficiente (no dispara reglas)",
        "hechos": {"tamano_grande":0.9,"orejas_paradas":0.9},
        "top_esperado": None,
    },
    {
        "nombre": "13) Husky parcial sin ojos azules (no dispara)",
        "hechos": {"tamano_grande":0.9,"pelo_largo":0.85,"orejas_paradas":0.9,"cola_poblada":0.8},
        "top_esperado": None,
    },
    {
        "nombre": "14) Golden vs Labrador (ambos disparan)",
        "hechos": {"tamano_grande":0.9,"dorado":0.9,"pelo_medio":0.7,"pelo_corto":0.7,"orejas_caidas":0.8},
        "top_esperado": "Golden Retriever",
    },
]


def run_tests_basicos():
    ok = 0
    for t in TESTS:
        ranking, logs = inferir_1pasada(t["hechos"], REGLAS)
        top = ranking[0][0] if ranking else None
        cf  = ranking[0][1] if ranking else None

        passed = (top == t["top_esperado"])
        status = "✅" if passed else "❌"
        print(f"{status} {t['nombre']} -> top={top} cf={None if cf is None else round(cf,3)}")

        if not passed:
            print("   esperado:", t["top_esperado"])
            print("   ranking:", [(r, round(c,3)) for r,c in ranking[:5]])
        else:
            ok += 1

    print(f"\nResultado básicos: {ok}/{len(TESTS)} tests OK")


def run_tests_unitarios_ranking_y_respaldo():
    ok = 0
    total = 0

    def check(condicion, nombre, detalle_error=""):
        nonlocal ok, total
        total += 1
        if condicion:
            ok += 1
            print(f"✅ {nombre}")
        else:
            print(f"❌ {nombre}")
            if detalle_error:
                print(f"   {detalle_error}")

    # U1: el ranking queda ordenado de mayor a menor CF
    hechos_u1 = {"tamano_grande":0.9,"dorado":0.9,"pelo_medio":0.7,"pelo_corto":0.7,"orejas_caidas":0.8}
    ranking_u1, _ = inferir_1pasada(hechos_u1, REGLAS)
    ordenado = all(ranking_u1[i][1] >= ranking_u1[i+1][1] for i in range(len(ranking_u1)-1))
    check(ordenado, "U1 Ranking ordenado por CF descendente")

    # U2: sin respaldo, caso parcial no retorna ranking
    hechos_u2 = {"tamano_grande":0.9, "pelo_largo":0.85, "orejas_paradas":0.9, "cola_poblada":0.8}
    ranking_u2, _ = inferir_1pasada(hechos_u2, REGLAS, usar_respaldo=False, min_coincidencias=2)
    check(len(ranking_u2) == 0, "U2 Sin respaldo no hay ranking para coincidencias parciales")

    # U3: con respaldo, mismo caso sí retorna ranking
    ranking_u3, logs_u3 = inferir_1pasada(hechos_u2, REGLAS, usar_respaldo=True, min_coincidencias=2)
    check(len(ranking_u3) > 0, "U3 Con respaldo sí hay ranking en coincidencias parciales")

    # U4: verificar que se usó método de respaldo en logs de la mejor raza
    if ranking_u3:
        mejor_u3 = ranking_u3[0][0]
        metodos_u3 = [entrada.get("metodo") for entrada in logs_u3.get(mejor_u3, [])]
        check("respaldo_parcial" in metodos_u3, "U4 Logs indican uso de respaldo_parcial")
    else:
        check(False, "U4 Logs indican uso de respaldo_parcial", "No hubo ranking para inspeccionar logs")

    # U5: sensibilidad a min_coincidencias con un único rasgo
    hechos_u5 = {"orejas_paradas": 0.9}
    ranking_u5_a, _ = inferir_1pasada(hechos_u5, REGLAS, usar_respaldo=True, min_coincidencias=2)
    ranking_u5_b, _ = inferir_1pasada(hechos_u5, REGLAS, usar_respaldo=True, min_coincidencias=1)
    check(len(ranking_u5_a) == 0 and len(ranking_u5_b) > 0,
          "U5 min_coincidencias controla activación del respaldo",
          f"len(min=2)={len(ranking_u5_a)}, len(min=1)={len(ranking_u5_b)}")

    # U6: el top de respaldo parcial no debe ser None
    top_u6 = ranking_u3[0][0] if ranking_u3 else None
    check(top_u6 is not None, "U6 Top del ranking parcial definido")

    print(f"\nResultado unitarios: {ok}/{total} tests OK")


print("=== BLOQUE A: TESTS BÁSICOS ===")
run_tests_basicos()
print("\n=== BLOQUE B: TESTS UNITARIOS (RANKING + RESPALDO) ===")
run_tests_unitarios_ranking_y_respaldo()

=== BLOQUE A: TESTS BÁSICOS ===
✅ 1) Husky claro -> top=Husky Siberiano cf=0.917
✅ 2) Pastor Alemán claro -> top=Pastor Aleman cf=0.736
✅ 3) Beagle claro -> top=Beagle cf=0.72
✅ 4) Golden claro -> top=Golden Retriever cf=0.744
✅ 5) Rottweiler claro -> top=Rottweiler cf=0.752
✅ 6) Dachshund claro -> top=Dachshund cf=0.728
✅ 7) Bulldog Francés claro -> top=Bulldog Frances cf=0.744
✅ 8) Poodle claro (con color_uniforme) -> top=Poodle cf=0.736
✅ 9) Labrador claro -> top=Labrador Retriever cf=0.728
✅ 10) Conflicto Labrador (rasgos de labrador + pelo_rizado) -> top=Labrador Retriever cf=0.382
✅ 11) Ambiguo: Poodle vs Dachshund (+ Labrador negativo por rizado) -> top=Dachshund cf=0.637
✅ 12) Insuficiente (no dispara reglas) -> top=None cf=None
✅ 13) Husky parcial sin ojos azules (no dispara) -> top=None cf=None
✅ 14) Golden vs Labrador (ambos disparan) -> top=Golden Retriever cf=0.651

Resultado básicos: 14/14 tests OK

=== BLOQUE B: TESTS UNITARIOS (RANKING + RESPALDO) ===
✅ U1 Ranking orden

## 🎮 Interfaz Interactiva del Usuario

Sistema interactivo con 4 modos de entrada:
1. 🎤 **Micrófono**: Describe verbalmente al perro
2. ⌨️ **Texto manual**: Descripción con palabras clave  
3. 📚 **Ejemplos**: Casos predefinidos
4. 🎯 **Rasgos personalizados**: Ingreso manual con certezas

In [27]:
# =========================
# 🎮 INTERFAZ INTERACTIVA DEL USUARIO
# =========================

def menu_principal():
    print("\n" + "="*70)
    print("🐶 SISTEMA EXPERTO DE IDENTIFICACIÓN DE RAZAS DE PERROS")
    print("="*70)
    print("\nElige cómo deseas proporcionar información:")
    print("\n  1️⃣  Usar MICRÓFONO (describe verbalmente al perro)")
    print("  2️⃣  Ingresar TEXTO MANUAL (describe con palabras clave)")
    print("  3️⃣  Cargar EJEMPLO PREDEFINIDO")
    print("  4️⃣  Ingresar RASGOS PERSONALIZADOS (con certeza)")
    print("\n" + "="*70)

    while True:
        try:
            opcion = input("\n¿Qué opción prefieres? (1-4): ").strip()
            if opcion in ["1", "2", "3", "4"]:
                return opcion
            print("❌ Opción no válida. Por favor, elige 1, 2, 3 ó 4.")
        except KeyboardInterrupt:
            print("\n\n👋 Hasta luego!")
            return None


def opcion_1_microfono():
    print("\n🎤 Usando MICRÓFONO...")
    resultado = diagnosticar_desde_audio(usando_1pasada=True, top_n=5)
    return resultado


def opcion_2_texto():
    print("\n⌨️  Ingresa una descripción del perro (ejemplo: 'es grande, pelo largo, ojos azules, orejas paradas'):")
    texto = input("Descripción: ").strip()

    if not texto:
        print("❌ No ingresaste nada.")
        return None

    hechos_audio = extraer_hechos_desde_texto(texto)

    if not hechos_audio:
        print("❌ No se detectaron rasgos válidos en tu descripción.")
        return None

    print("\n🔎 Rasgos detectados:")
    for h, cf in sorted(hechos_audio.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {h}: {cf:.2f}")

    ranking, logs = inferir_1pasada(hechos_audio, REGLAS, usar_respaldo=True, min_coincidencias=2)

    print("\n📊 TOP 5 RAZAS IDENTIFICADAS:")
    if not ranking:
        print("  ⚠️  Diagnóstico no concluyente")
        print("  💡 Agrega más rasgos (tamaño + pelo + orejas o color).")
        return None

    for i, (raza, cf) in enumerate(ranking[:5], 1):
        barra = "▓" * int(cf * 25)
        print(f"  {i}. {raza:30s} [{cf:6.3f}] {barra}")

    mejor_raza, mejor_cf = ranking[0]
    print(f"\n✅ RECOMENDACIÓN: {mejor_raza} (CF = {mejor_cf:.3f})")

    return {
        "texto": texto,
        "hechos": hechos_audio,
        "ranking": ranking,
        "logs": logs,
    }


def opcion_3_ejemplos():
    print("\n📚 Ejemplos predefinidos:\n")
    print("  A) Husky Siberiano claro")
    print("  B) Bulldog Francés claro")
    print("  C) Dalmatian claro")

    while True:
        ejemplo = input("\n¿Cuál ejemplo? (A/B/C): ").strip().upper()
        if ejemplo in ["A", "B", "C"]:
            break
        print("❌ Opción no válida.")

    ejemplos = {
        "A": ("Husky Siberiano", hechos_ejemplo_1),
        "B": ("Bulldog Francés", hechos_ejemplo_2),
        "C": ("Dalmatian", hechos_ejemplo_3),
    }

    nombre, hechos_ej = ejemplos[ejemplo]
    print(f"\n📌 Usando ejemplo: {nombre}")
    print(f"   Rasgos: {list(hechos_ej.keys())}")

    ranking, logs = inferir_1pasada(hechos_ej, REGLAS, usar_respaldo=True, min_coincidencias=2)

    print("\n📊 TOP 5 RAZAS INFERIDAS:")
    if not ranking:
        print("  ⚠️  Diagnóstico no concluyente")
        return None

    for i, (raza, cf) in enumerate(ranking[:5], 1):
        barra = "▓" * int(cf * 25)
        print(f"  {i}. {raza:30s} [{cf:6.3f}] {barra}")

    mejor_raza, mejor_cf = ranking[0]
    print(f"\n✅ RECOMENDACIÓN: {mejor_raza} (CF = {mejor_cf:.3f})")

    return {
        "ejemplo": nombre,
        "hechos": hechos_ej,
        "ranking": ranking,
        "logs": logs,
    }


def opcion_4_personalizado():
    print("\n🎯 INGRESO PERSONALIZADO DE RASGOS")
    print("Rasgos disponibles:")

    rasgos_list = list(RASGOS_DISPONIBLES.keys())
    for i, rasgo in enumerate(rasgos_list, 1):
        print(f"  {i:2d}. {rasgo:30s} - {RASGOS_DISPONIBLES[rasgo]}")

    hechos_custom = {}

    print("\n➕ Agrega rasgos (escribe el número y la certeza 0-1).")
    print("   Formato: '5 0.9' = rasgo #5 con certeza 0.9")
    print("   Escribe 'listo' cuando termines.\n")

    while True:
        entrada = input("Rasgo (número certeza) o 'listo': ").strip().lower()

        if entrada == "listo":
            break

        try:
            partes = entrada.split()
            if len(partes) != 2:
                print("❌ Formato incorrecto. Intenta: '5 0.9'")
                continue

            idx = int(partes[0]) - 1
            cf_val = float(partes[1])

            if idx < 0 or idx >= len(rasgos_list):
                print("❌ Número de rasgo inválido.")
                continue

            if not (0 <= cf_val <= 1):
                print("❌ Certeza debe estar entre 0 y 1.")
                continue

            rasgo = rasgos_list[idx]
            hechos_custom[rasgo] = cf_val
            print(f"✅ Agregado: {rasgo} = {cf_val}")

        except ValueError:
            print("❌ Formato incorrecto. Intenta: '5 0.9'")

    if not hechos_custom:
        print("❌ No agregaste rasgos.")
        return None

    print("\n🔎 Rasgos ingresados:")
    for h, cf in sorted(hechos_custom.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {h:30s}: {cf:.2f}")

    ranking, logs = inferir_1pasada(hechos_custom, REGLAS, usar_respaldo=True, min_coincidencias=2)

    print("\n📊 TOP 5 RAZAS INFERIDAS:")
    if not ranking:
        print("  ⚠️  Diagnóstico no concluyente")
        print("  💡 Intenta ingresar al menos 3 rasgos con certeza >= 0.7")
        return None

    for i, (raza, cf) in enumerate(ranking[:5], 1):
        barra = "▓" * int(cf * 25)
        print(f"  {i}. {raza:30s} [{cf:6.3f}] {barra}")

    mejor_raza, mejor_cf = ranking[0]
    print(f"\n✅ RECOMENDACIÓN: {mejor_raza} (CF = {mejor_cf:.3f})")

    return {
        "hechos": hechos_custom,
        "ranking": ranking,
        "logs": logs,
    }


def mostrar_trazabilidad(resultado):
    if not resultado or "logs" not in resultado:
        return

    print("\n" + "="*70)
    print("📋 TRAZABILIDAD DE LA INFERENCIA")
    print("="*70)

    logs = resultado["logs"]
    ranking = resultado["ranking"]

    if not ranking:
        return

    mejor_raza = ranking[0][0]

    if mejor_raza in logs:
        print(f"\n🔍 Reglas que contribuyeron a: {mejor_raza}\n")
        for i, log in enumerate(logs[mejor_raza], 1):
            metodo = log.get("metodo", "regla_completa")
            print(f"Regla #{i} ({log['regla']}) - {metodo}:")
            print(f"  SI: {', '.join(log['si'])}")
            if "si_coincidencias" in log and log["si_coincidencias"]:
                print(f"  Coincidencias: {', '.join(log['si_coincidencias'])}")
            if "cobertura" in log:
                print(f"  Cobertura: {log['cobertura']:.3f}")
            print(f"  FC antecedentes: {log['fc_antecedentes']}")
            print(f"  FC regla: {log['fc_regla']}")
            print(f"  Aporte: {log['fc_aporte']}")
            print(f"  CF: {log['cf_prev']:.3f} → {log['cf_nuevo']:.3f}")
            print()


def run_interactivo():
    """Ejecuta el flujo interactivo del sistema experto"""
    try:
        while True:
            opcion = menu_principal()

            if opcion is None:
                break

            resultado = None

            if opcion == "1":
                resultado = opcion_1_microfono()
            elif opcion == "2":
                resultado = opcion_2_texto()
            elif opcion == "3":
                resultado = opcion_3_ejemplos()
            elif opcion == "4":
                resultado = opcion_4_personalizado()

            if resultado:
                mostrar_trazabilidad(resultado)

            continuar = input("\n¿Deseas analizar otro perro? (s/n): ").strip().lower()
            if continuar != "s":
                print("\n👋 ¡Gracias por usar el sistema experto de razas de perros!")
                break

    except KeyboardInterrupt:
        print("\n\n👋 Programa interrumpido. ¡Hasta luego!")


# INICIAR INTERFAZ INTERACTIVA:
print("✅ Sistema cargado. Para usar, ejecuta:")
print("   run_interactivo()")
print("\nO prueba opciones individuales:")
print("   opcion_1_microfono()  # diagnóstico por voz")
print("   opcion_2_texto()  # texto manual")
print("   opcion_3_ejemplos()  # ejemplos")
print("   opcion_4_personalizado()  # rasgos personalizados")


✅ Sistema cargado. Para usar, ejecuta:
   run_interactivo()

O prueba opciones individuales:
   opcion_1_microfono()  # diagnóstico por voz
   opcion_2_texto()  # texto manual
   opcion_3_ejemplos()  # ejemplos
   opcion_4_personalizado()  # rasgos personalizados


In [28]:
# PRUEBA RÁPIDA POR TEXTO (flujo equivalente a opción 2)
texto_prueba = "es un perro grande, de pelo largo, orejas paradas y cola peluda"
print("Texto de prueba:", texto_prueba)

hechos_prueba = extraer_hechos_desde_texto(texto_prueba)
print("\nHechos detectados:")
for h, cf in sorted(hechos_prueba.items(), key=lambda x: x[1], reverse=True):
    print(f"- {h}: {cf:.2f}")

ranking_prueba, logs_prueba = inferir_1pasada(
    hechos_prueba,
    REGLAS,
    usar_respaldo=True,
    min_coincidencias=2,
)

print("\nTOP 5 por texto:")
if not ranking_prueba:
    print("- Diagnóstico no concluyente")
else:
    for i, (raza, cf) in enumerate(ranking_prueba[:5], 1):
        print(f"{i}. {raza} -> CF={cf:.3f}")
    print(f"\nRecomendación principal: {ranking_prueba[0][0]} (CF={ranking_prueba[0][1]:.3f})")

Texto de prueba: es un perro grande, de pelo largo, orejas paradas y cola peluda

Hechos detectados:
- tamano_grande: 0.90
- pelo_largo: 0.90
- orejas_paradas: 0.90
- cola_poblada: 0.85

TOP 5 por texto:
1. Husky Siberiano -> CF=0.646
2. Pastor Belga -> CF=0.601
3. Collie -> CF=0.516
4. Pastor Aleman -> CF=0.469
5. Akita -> CF=0.401

Recomendación principal: Husky Siberiano (CF=0.646)


## 🧪 PRUEBAS DE LA INTERFAZ DE USUARIO

Probamos las funcionalidades principales:
- ✅ Opción 1: Reconocimiento de voz
- ✅ Opción 2: Texto manual
- ✅ Opción 3: Ejemplos predefinidos  
- ✅ Opción 4: Rasgos personalizados

In [29]:
# =========================
# PRUEBA 1: OPCIÓN 1 - RECONOCIMIENTO DE VOZ
# =========================
print("="*70)
print("🧪 PRUEBA 1: OPCIÓN 1 - RECONOCIMIENTO DE VOZ")
print("="*70)
print("\n📋 INSTRUCCIONES:")
print("   1. Asegúrate de que tu micrófono esté conectado y funcionando")
print("   2. Verifica que Python tenga permisos de micrófono en Windows")
print("   3. Cuando ejecutes esta celda, esperará que hables")
print("   4. Di algo simple como: 'perro grande dorado'")
print("   5. Presiona ENTER cuando termines de hablar")
print("\n⚠️  Si hay errores, revisa:")
print("   - PyAudio instalado: pip install pyaudio")
print("   - Permisos de micrófono en Configuración > Privacidad > Micrófono")
print("   - Micrófono seleccionado como dispositivo predeterminado")
print("\n" + "="*70)

# Probar reconocimiento básico
print("\n🔴 Iniciando prueba de reconocimiento...\n")

try:
    texto_reconocido = reconocer_voz_es(timeout=10, phrase_time_limit=15)
    
    if texto_reconocido:
        print("\n" + "="*70)
        print("✅ ¡RECONOCIMIENTO EXITOSO!")
        print("="*70)
        print(f"\n📝 Texto capturado: '{texto_reconocido}'")
        
        # Extraer hechos
        hechos_voz = extraer_hechos_desde_texto(texto_reconocido)
        
        if hechos_voz:
            print(f"\n🔎 Se detectaron {len(hechos_voz)} rasgos:")
            for h, cf in sorted(hechos_voz.items(), key=lambda x: x[1], reverse=True):
                print(f"   • {h}: {cf:.2f}")
            
            # Inferencia
            print("\n📊 Ejecutando inferencia...")
            ranking, logs = inferir_1pasada(hechos_voz, REGLAS, usar_respaldo=True, min_coincidencias=2)
            
            if ranking:
                print("\n🏆 TOP 3 RAZAS:")
                for i, (raza, cf) in enumerate(ranking[:3], 1):
                    barra = "▓" * int(cf * 20)
                    print(f"  {i}. {raza:30s} [{cf:.3f}] {barra}")
                print(f"\n✅ RECOMENDACIÓN: {ranking[0][0]} (CF={ranking[0][1]:.3f})")
        else:
            print("\n⚠️  No se detectaron rasgos de perros")
            print("   Usa palabras como: grande, pequeño, dorado, negro, pelo largo, etc.")
    else:
        print("\n" + "="*70)
        print("❌ NO SE DETECTÓ VOZ O HUBO UN ERROR")
        print("="*70)
        print("\n💡 Posibles causas:")
        print("   • Micrófono no está conectado o seleccionado")
        print("   • No hay permisos de micrófono")
        print("   • PyAudio no está instalado correctamente")
        print("   • No hablaste lo suficientemente fuerte")
        
except Exception as e:
    print("\n" + "="*70)
    print("❌ ERROR EN LA PRUEBA")
    print("="*70)
    print(f"\nError: {e}")
    print("\n🔧 Solución recomendada:")
    print("   Usa la Opción 2 (Texto Manual) de la interfaz que funciona sin micrófono")
    import traceback
    print("\n📋 Detalles técnicos:")
    traceback.print_exc()

print("\n" + "="*70)

🧪 PRUEBA 1: OPCIÓN 1 - RECONOCIMIENTO DE VOZ

📋 INSTRUCCIONES:
   1. Asegúrate de que tu micrófono esté conectado y funcionando
   2. Verifica que Python tenga permisos de micrófono en Windows
   3. Cuando ejecutes esta celda, esperará que hables
   4. Di algo simple como: 'perro grande dorado'
   5. Presiona ENTER cuando termines de hablar

⚠️  Si hay errores, revisa:
   - PyAudio instalado: pip install pyaudio
   - Permisos de micrófono en Configuración > Privacidad > Micrófono
   - Micrófono seleccionado como dispositivo predeterminado


🔴 Iniciando prueba de reconocimiento...

🎤 Describe al perro (tamaño, pelo, orejas, color, etc.)...
   📊 Calibrando micrófono...
   🔴 GRABANDO (máximo 15s)... ¡Habla ahora!
   💬 Ejemplo: 'es un perro grande, dorado, con pelo largo'
   ⏸️  Procesando audio...
   ✅ Audio capturado correctamente

📝 Texto reconocido: "cola larga"

✅ ¡RECONOCIMIENTO EXITOSO!

📝 Texto capturado: 'cola larga'

⚠️  No se detectaron rasgos de perros
   Usa palabras como: gra

In [30]:
# =========================
# PRUEBA 2: OPCIÓN 2 - TEXTO MANUAL
# =========================
print("="*70)
print("🧪 PRUEBA 2: OPCIÓN 2 - TEXTO MANUAL")
print("="*70)

# Simulamos diferentes descripciones de texto
casos_texto = [
    "es un perro grande, dorado, pelo medio, orejas caídas",
    "mediano, pelo corto, orejas caidas, tricolor",
    "pequeño, hocico corto, cara plana, orejas paradas, robusto",
]

for i, texto in enumerate(casos_texto, 1):
    print(f"\n--- Caso {i} ---")
    print(f"📝 Texto: '{texto}'")
    
    hechos = extraer_hechos_desde_texto(texto)
    
    if not hechos:
        print("❌ No se detectaron rasgos válidos")
        continue
    
    print("\n🔎 Rasgos detectados:")
    for h, cf in sorted(hechos.items(), key=lambda x: x[1], reverse=True):
        print(f"  - {h}: {cf:.2f}")
    
    ranking, logs = inferir_1pasada(hechos, REGLAS, usar_respaldo=True, min_coincidencias=2)
    
    print("\n📊 TOP 3 RAZAS:")
    if not ranking:
        print("  ⚠️  Diagnóstico no concluyente")
    else:
        for j, (raza, cf) in enumerate(ranking[:3], 1):
            barra = "▓" * int(cf * 20)
            print(f"  {j}. {raza:30s} [{cf:.3f}] {barra}")
        print(f"\n✅ Recomendación: {ranking[0][0]} (CF={ranking[0][1]:.3f})")

print("\n" + "="*70)

🧪 PRUEBA 2: OPCIÓN 2 - TEXTO MANUAL

--- Caso 1 ---
📝 Texto: 'es un perro grande, dorado, pelo medio, orejas caídas'

🔎 Rasgos detectados:
  - tamano_grande: 0.90
  - orejas_caidas: 0.90
  - dorado: 0.90
  - pelo_medio: 0.80

📊 TOP 3 RAZAS:
  1. Golden Retriever               [0.744] ▓▓▓▓▓▓▓▓▓▓▓▓▓▓

✅ Recomendación: Golden Retriever (CF=0.744)

--- Caso 2 ---
📝 Texto: 'mediano, pelo corto, orejas caidas, tricolor'

🔎 Rasgos detectados:
  - pelo_corto: 0.90
  - orejas_caidas: 0.90
  - tricolor: 0.90
  - tamano_mediano: 0.85

📊 TOP 3 RAZAS:
  1. Beagle                         [0.765] ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

✅ Recomendación: Beagle (CF=0.765)

--- Caso 3 ---
📝 Texto: 'pequeño, hocico corto, cara plana, orejas paradas, robusto'

🔎 Rasgos detectados:
  - tamano_pequeno: 0.90
  - orejas_paradas: 0.90
  - hocico_corto: 0.90
  - robusto: 0.82

📊 TOP 3 RAZAS:
  1. Bulldog Frances                [0.628] ▓▓▓▓▓▓▓▓▓▓▓▓
  2. Shiba Inu                      [0.528] ▓▓▓▓▓▓▓▓▓▓

✅ Recomendación: Bulldog France

In [31]:
# =========================
# PRUEBA 3: OPCIÓN 3 - EJEMPLOS PREDEFINIDOS
# =========================
print("="*70)
print("🧪 PRUEBA 3: OPCIÓN 3 - EJEMPLOS PREDEFINIDOS")
print("="*70)

ejemplos = {
    "A": ("Husky Siberiano", hechos_ejemplo_1),
    "B": ("Bulldog Frances", hechos_ejemplo_2),
    "C": ("Dalmatian", hechos_ejemplo_3),
}

for letra, (nombre, hechos_ej) in ejemplos.items():
    print(f"\n--- Ejemplo {letra}: {nombre} ---")
    print(f"   Rasgos: {', '.join(list(hechos_ej.keys())[:5])}...")
    
    ranking, logs = inferir_1pasada(hechos_ej, REGLAS, usar_respaldo=True, min_coincidencias=2)
    
    print("\n📊 TOP 3 RAZAS:")
    if not ranking:
        print("  ⚠️  Diagnóstico no concluyente")
    else:
        for i, (raza, cf) in enumerate(ranking[:3], 1):
            barra = "▓" * int(cf * 20)
            emoji = "🎯" if i == 1 else "  "
            print(f"  {emoji} {i}. {raza:30s} [{cf:.3f}] {barra}")
        
        mejor_raza, mejor_cf = ranking[0]
        es_correcto = nombre.lower() in mejor_raza.lower() or mejor_raza.lower() in nombre.lower()
        status = "✅ CORRECTO" if es_correcto else "⚠️  VERIFICAR"
        print(f"\n{status}: Esperado='{nombre}', Obtenido='{mejor_raza}' (CF={mejor_cf:.3f})")

print("\n" + "="*70)

🧪 PRUEBA 3: OPCIÓN 3 - EJEMPLOS PREDEFINIDOS

--- Ejemplo A: Husky Siberiano ---
   Rasgos: tamano_grande, pelo_largo, ojos_azules, orejas_paradas, cola_poblada...

📊 TOP 3 RAZAS:
  🎯 1. Husky Siberiano                [0.857] ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

✅ CORRECTO: Esperado='Husky Siberiano', Obtenido='Husky Siberiano' (CF=0.857)

--- Ejemplo B: Bulldog Frances ---
   Rasgos: tamano_pequeno, cara_plana, hocico_corto, orejas_paradas, arrugas_cara...

📊 TOP 3 RAZAS:
  🎯 1. Bulldog Frances                [0.941] ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

✅ CORRECTO: Esperado='Bulldog Frances', Obtenido='Bulldog Frances' (CF=0.941)

--- Ejemplo C: Dalmatian ---
   Rasgos: tamano_grande, color_blanco_manchas_negras, manchas_oscuras, pelaje_corto, orejas_caidas...

📊 TOP 3 RAZAS:
  🎯 1. Dalmatian                      [0.930] ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

✅ CORRECTO: Esperado='Dalmatian', Obtenido='Dalmatian' (CF=0.930)



In [32]:
# =========================
# PRUEBA 4: OPCIÓN 4 - RASGOS PERSONALIZADOS
# =========================
print("="*70)
print("🧪 PRUEBA 4: OPCIÓN 4 - RASGOS PERSONALIZADOS")
print("="*70)

# Simulamos diferentes configuraciones personalizadas
casos_personalizados = [
    {
        "nombre": "Beagle típico",
        "hechos": {
            "tamano_mediano": 0.9,
            "pelo_corto": 0.95,
            "orejas_caidas": 0.90,
            "tricolor": 0.85,
            "hocico_puntiagudo": 0.80,
        },
        "esperado": "Beagle"
    },
    {
        "nombre": "Poodle rizado",
        "hechos": {
            "tamano_mediano": 0.85,
            "pelo_rizado": 0.95,
            "orejas_caidas": 0.90,
            "color_uniforme": 0.85,
            "hipoalergenico": 0.80,
        },
        "esperado": "Poodle"
    },
    {
        "nombre": "Chihuahua mini",
        "hechos": {
            "tamano_muy_pequeno": 0.95,
            "cabeza_grande": 0.90,
            "ojos_grandes": 0.88,
            "orejas_puntiagudas": 0.85,
        },
        "esperado": "Chihuahua"
    },
]

for i, caso in enumerate(casos_personalizados, 1):
    print(f"\n--- Caso {i}: {caso['nombre']} ---")
    
    print("🔎 Rasgos ingresados:")
    for h, cf in sorted(caso['hechos'].items(), key=lambda x: x[1], reverse=True):
        print(f"  - {h:30s}: {cf:.2f}")
    
    ranking, logs = inferir_1pasada(caso['hechos'], REGLAS, usar_respaldo=True, min_coincidencias=2)
    
    print("\n📊 TOP 3 RAZAS:")
    if not ranking:
        print("  ⚠️  Diagnóstico no concluyente")
    else:
        for j, (raza, cf) in enumerate(ranking[:3], 1):
            barra = "▓" * int(cf * 20)
            emoji = "🎯" if j == 1 else "  "
            print(f"  {emoji} {j}. {raza:30s} [{cf:.3f}] {barra}")
        
        mejor_raza, mejor_cf = ranking[0]
        es_correcto = caso['esperado'].lower() in mejor_raza.lower()
        status = "✅ CORRECTO" if es_correcto else "⚠️  VERIFICAR"
        print(f"\n{status}: Esperado='{caso['esperado']}', Obtenido='{mejor_raza}' (CF={mejor_cf:.3f})")
        
        # Mostrar trazabilidad resumida para el top 1
        if mejor_raza in logs:
            print(f"\n📋 Trazabilidad resumida ({len(logs[mejor_raza])} reglas):")
            for log in logs[mejor_raza][:2]:  # Solo mostrar las primeras 2 reglas
                metodo = log.get("metodo", "regla_completa")
                print(f"  • {log['regla']} ({metodo}): aporte={log['fc_aporte']:.3f}, CF final={log['cf_nuevo']:.3f}")

print("\n" + "="*70)

🧪 PRUEBA 4: OPCIÓN 4 - RASGOS PERSONALIZADOS

--- Caso 1: Beagle típico ---
🔎 Rasgos ingresados:
  - pelo_corto                    : 0.95
  - tamano_mediano                : 0.90
  - orejas_caidas                 : 0.90
  - tricolor                      : 0.85
  - hocico_puntiagudo             : 0.80

📊 TOP 3 RAZAS:
  🎯 1. Beagle                         [0.765] ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

✅ CORRECTO: Esperado='Beagle', Obtenido='Beagle' (CF=0.765)

📋 Trazabilidad resumida (1 reglas):
  • R7 (regla_completa): aporte=0.765, CF final=0.765

--- Caso 2: Poodle rizado ---
🔎 Rasgos ingresados:
  - pelo_rizado                   : 0.95
  - orejas_caidas                 : 0.90
  - tamano_mediano                : 0.85
  - color_uniforme                : 0.85
  - hipoalergenico                : 0.80

📊 TOP 3 RAZAS:
  🎯 1. Poodle                         [0.935] ▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓

✅ CORRECTO: Esperado='Poodle', Obtenido='Poodle' (CF=0.935)

📋 Trazabilidad resumida (2 reglas):
  • R25 (regla_completa): aporte

In [33]:
# =========================
# PRUEBA 5: FUNCIONALIDAD DE TRAZABILIDAD COMPLETA
# =========================
print("="*70)
print("🧪 PRUEBA 5: TRAZABILIDAD COMPLETA")
print("="*70)

# Caso con reglas completas
print("\n--- Caso A: Reglas completas (Husky Siberiano) ---")
hechos_test = {
    "tamano_grande": 0.9,
    "pelo_largo": 0.85,
    "ojos_azules": 0.8,
    "orejas_paradas": 0.9,
    "cola_poblada": 0.8
}

ranking_test, logs_test = inferir_1pasada(hechos_test, REGLAS, usar_respaldo=True, min_coincidencias=2)

if ranking_test:
    mejor_raza = ranking_test[0][0]
    print(f"\n🔍 Trazabilidad completa para: {mejor_raza}\n")
    
    for i, log in enumerate(logs_test[mejor_raza], 1):
        metodo = log.get("metodo", "regla_completa")
        print(f"Regla #{i} ({log['regla']}) - {metodo}:")
        print(f"  SI: {', '.join(log['si'])}")
        if "si_coincidencias" in log and log["si_coincidencias"] != log['si']:
            print(f"  Coincidencias: {', '.join(log['si_coincidencias'])}")
        if "cobertura" in log:
            print(f"  Cobertura: {log['cobertura']:.3f}")
        print(f"  FC antecedentes: {log['fc_antecedentes']}")
        print(f"  FC regla: {log['fc_regla']}")
        print(f"  Aporte: {log['fc_aporte']:.3f}")
        print(f"  CF: {log['cf_prev']:.3f} → {log['cf_nuevo']:.3f}")
        print()

# Caso con respaldo parcial
print("\n--- Caso B: Respaldo parcial (sin reglas completas) ---")
hechos_parcial = {
    "tamano_grande": 0.9,
    "pelo_largo": 0.85,
    "orejas_paradas": 0.9,
}

ranking_parcial, logs_parcial = inferir_1pasada(hechos_parcial, REGLAS, usar_respaldo=True, min_coincidencias=2)

if ranking_parcial:
    mejor_raza_p = ranking_parcial[0][0]
    print(f"🔍 Trazabilidad parcial para: {mejor_raza_p}")
    print(f"   Total de reglas parciales: {len(logs_parcial[mejor_raza_p])}\n")
    
    # Mostrar solo las primeras 3 reglas parciales
    for i, log in enumerate(logs_parcial[mejor_raza_p][:3], 1):
        metodo = log.get("metodo", "regla_completa")
        print(f"Regla #{i} ({log['regla']}) - {metodo}:")
        print(f"  SI: {', '.join(log['si'])}")
        print(f"  Coincidencias: {', '.join(log['si_coincidencias'])}")
        print(f"  Cobertura: {log.get('cobertura', 'N/A')}")
        print(f"  Aporte: {log['fc_aporte']:.3f}, CF final: {log['cf_nuevo']:.3f}")
        print()
else:
    print("⚠️  No hubo ranking con respaldo")

print("="*70)

🧪 PRUEBA 5: TRAZABILIDAD COMPLETA

--- Caso A: Reglas completas (Husky Siberiano) ---

🔍 Trazabilidad completa para: Husky Siberiano

Regla #1 (R4) - regla_completa:
  SI: tamano_grande, pelo_largo, ojos_azules, orejas_paradas, cola_poblada
  FC antecedentes: 0.8
  FC regla: 0.95
  Aporte: 0.760
  CF: 0.000 → 0.760

Regla #2 (R6) - regla_completa:
  SI: ojos_azules, orejas_paradas
  FC antecedentes: 0.8
  FC regla: 0.82
  Aporte: 0.656
  CF: 0.760 → 0.917


--- Caso B: Respaldo parcial (sin reglas completas) ---
🔍 Trazabilidad parcial para: Pastor Belga
   Total de reglas parciales: 1

Regla #1 (RESP_R55) - respaldo_parcial:
  SI: tamano_grande, pelo_largo, color_negro, orejas_paradas
  Coincidencias: tamano_grande, pelo_largo, orejas_paradas
  Cobertura: 0.75
  Aporte: 0.567, CF final: 0.567



## 📚 Ejemplos de Entrada por Cada Opción

### **Opción 1️⃣: Usando MICRÓFONO**

Define verbalmente al perro. El sistema reconocerá español. Ejemplos de lo que puedes decir:

- *"Es un perro grande, con pelo largo y ojos azules, orejas paradas y cola peluda"*
- *"Mediano, pelo corto, orejas caídas, tricolor: blanco, negro y fuego"*
- *"Pequeño, cara plana, orejas paradas, cuerpo robusto"*

El sistema extrae automáticamente los rasgos y calcula factor de certeza.

---

### **Opción 2️⃣: Ingresar TEXTO MANUAL**

Escribe una descripción con palabras clave. Ejemplos:

```
grande, pelo largo, ojos azules, orejas paradas, cola poblada
```

```
mediano, pelo corto, orejas caidas, tricolor
```

```
pequeño, hocico corto, cara plana, robusto
```

**Palabras clave reconocidas:**  
- Tamaño: `grande`, `mediano`, `pequeño`, `chico`, `enorme`, `gigante`, `mini`
- Pelo: `largo`, `corto`, `medio`, `rizado`, `ondulado`, `duro`
- Orejas: `paradas`, `caidas`, `largas`, `triangulares`, `pequeñas`
- Color: `dorado`, `negro`, `blanco`, `rojo`, `tricolor`, `bicolor`, `manchas`
- Otros: `musculoso`, `robusto`, `alargado`, `cara plana`, `negra`

---

### **Opción 3️⃣: Cargar EJEMPLO PREDEFINIDO**

El sistema tiene 3 ejemplos listos para probar:

| Opción | Raza | Características |
|--------|------|-----------------|
| **A** | Husky Siberiano | Grande, pelo largo, ojos azules, orejas paradas, cola poblada |
| **B** | Bulldog Francés | Pequeño, cara plana, hocico corto, orejas paradas, cuerpo robusto |
| **C** | Dalmatian | Grande, blanco con manchas negras, pelo corto |

Solo elige A, B o C cuando se te pida.

---

### **Opción 4️⃣: Ingresar RASGOS PERSONALIZADOS**

Selecciona rasgos individuales de la lista y asigna certeza (0.0 a 1.0).

**Ejemplo 1: Perro grande con rasgos claros**
```
1 0.9      # tamano_muy_pequeno (no, probabilidad baja)
4 0.9      # tamano_grande (sí, muy seguro)
17 0.8     # pelo_largo (bastante seguro)
20 0.9     # orejas_paradas (muy seguro)
23 0.85    # cola_poblada (seguro)
26 0.8     # ojos_azules (bastante seguro)
listo
```

**Ejemplo 2: Perro pequeño chato**
```
2 0.95     # tamano_pequeno (muy seguro)
10 0.9     # hocico_corto (muy seguro)
14 0.85    # cara_plana (seguro)
20 0.8     # orejas_paradas (bastante seguro)
46 0.85    # cuerpo_robusto (seguro)
listo
```

**Formato obligatorio:** `[número_rasgo] [certeza]`  
- Número: índice en lista (1-100+)
- Certeza: 0.0 (no seguro) a 1.0 (muy seguro)
- Termina con: `listo`

---

## ▶️ **COMIENZA AQUÍ**

In [34]:
# =================================================
# ▶️ EJECUTA ESTA CELDA PARA USAR LA INTERFAZ
# =================================================
run_interactivo()


🐶 SISTEMA EXPERTO DE IDENTIFICACIÓN DE RAZAS DE PERROS

Elige cómo deseas proporcionar información:

  1️⃣  Usar MICRÓFONO (describe verbalmente al perro)
  2️⃣  Ingresar TEXTO MANUAL (describe con palabras clave)
  3️⃣  Cargar EJEMPLO PREDEFINIDO
  4️⃣  Ingresar RASGOS PERSONALIZADOS (con certeza)


🎤 Usando MICRÓFONO...
🎤 Describe al perro (tamaño, pelo, orejas, color, etc.)...
   📊 Calibrando micrófono...
   🔴 GRABANDO (máximo 20s)... ¡Habla ahora!
   💬 Ejemplo: 'es un perro grande, dorado, con pelo largo'
   ⏸️  Procesando audio...
   ✅ Audio capturado correctamente

📝 Texto reconocido: "musculoso alargado cara plana"

🔎 Hechos detectados (con certeza):
- hocico_corto: 0.90
- musculoso: 0.84

📊 Top razas inferidas:
1. Boxer -> CF=0.370

✅ Recomendación principal: Boxer (CF=0.370)

📋 TRAZABILIDAD DE LA INFERENCIA

🔍 Reglas que contribuyeron a: Boxer

Regla #1 (RESP_R36) - respaldo_parcial:
  SI: tamano_grande, musculoso, hocico_corto, mandibula_pronunciada
  Coincidencias: musculos